# Module 03 — Lesson 03
# Percentiles and Correlation

---

> Where do you rank? Do two things move together? These are two of the most common questions in applied statistics — and the answers come from percentiles and correlation.

You get your JEE result: **92 percentile**. Not 92%. Percentile tells you something percentage never can — your rank relative to everyone else who took the exam. A 92 percentile means you outperformed 92% of all test-takers, even if your raw score was only 65%.

This lesson covers two ideas that show up constantly in data science:

- **Percentiles** — "where does this value rank?" (JEE/CAT rank, salary benchmarking, growth charts for babies, website load-time SLAs — "p95 latency")
- **Correlation** — "do these two variables move together?" (study hours vs marks, ad spend vs sales, temperature vs ice cream sales — the backbone of feature selection before you build any ML model)

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain the difference between a percentile and a percentage
- Compute any percentile from scratch using linear interpolation
- Compute quartiles, the IQR, and the five-number summary
- Detect outliers using the IQR method and compare it to the z-score method from Lesson 02
- Compute the Pearson correlation coefficient from scratch and interpret its strength and direction
- Explain why correlation does not imply causation, and why Pearson only captures linear relationships

---
## 1. Percentile vs Percentage — Why Rank Matters

In [ ]:
# Two different exams. Priya scored 65 on both.
exam_easy = [60, 65, 68, 70, 72, 75, 78, 80, 82, 85, 88, 90, 92, 94, 95, 96, 97, 98, 99, 100]
exam_hard = [20, 25, 30, 32, 35, 38, 40, 42, 45, 48, 50, 52, 55, 58, 60, 62, 65, 68, 70, 72]

priya_score = 65

rank_in_easy = sum(1 for x in exam_easy if x < priya_score)
rank_in_hard = sum(1 for x in exam_hard if x < priya_score)
n = len(exam_easy)

print(f"Priya scored {priya_score} on both exams.")
print()
print(f"Easy exam — {rank_in_easy}/{n} scored below her -> she's around the {rank_in_easy/n*100:.0f}th percentile")
print(f"Hard exam — {rank_in_hard}/{n} scored below her -> she's around the {rank_in_hard/n*100:.0f}th percentile")
print()
print("Same raw score. Very different standing. The percentage (65) never changes.")
print("The percentile — her RANK relative to everyone else — tells the real story.")

---
## 2. Computing Percentiles From Scratch

The **linear interpolation method** (the default used by NumPy, pandas, and Excel's `PERCENTILE.INC`) works like this:

1. Sort the data
2. Find the "rank position": $\text{rank} = \frac{p}{100} \times (n - 1)$
3. If the rank lands exactly on an index, return that value
4. If it lands between two indices, interpolate between them

This means percentiles don't require the value to already exist in the dataset — a percentile can fall *between* two data points.

In [ ]:
def percentile(data, p):
    '''
    Compute the p-th percentile of data using linear interpolation.
    p must be between 0 and 100.
    '''
    if not data:
        raise ValueError("Cannot compute percentile of an empty list")
    if not 0 <= p <= 100:
        raise ValueError("Percentile must be between 0 and 100")

    sorted_data = sorted(data)
    n = len(sorted_data)
    rank = (p / 100) * (n - 1)

    lower = int(rank)          # floor
    upper = min(lower + 1, n - 1)
    fraction = rank - lower

    return sorted_data[lower] + fraction * (sorted_data[upper] - sorted_data[lower])


scores = [55, 60, 62, 65, 68, 70, 72, 75, 78, 80, 82, 85, 88, 90, 95]

print(f"Scores (sorted): {sorted(scores)}")
print()
for p in [0, 25, 50, 75, 90, 95, 100]:
    print(f"  {p:>3}th percentile : {percentile(scores, p):.1f}")

print()
print("Note: 50th percentile == median. 0th percentile == min. 100th percentile == max.")

---
## 3. Quartiles and the Five-Number Summary

**Quartiles** split sorted data into four equal parts:

| Quartile | Percentile | Meaning |
|---|---|---|
| Q1 | 25th | 25% of data falls below this |
| Q2 | 50th | the median |
| Q3 | 75th | 75% of data falls below this |

The **Five-Number Summary** — min, Q1, median, Q3, max — is a compact description of a dataset's shape, and it's exactly what a box plot visualises.

The **Interquartile Range (IQR)** measures the spread of the *middle 50%* of the data, ignoring extremes:

$$IQR = Q3 - Q1$$

In [ ]:
def quartiles(data):
    '''Return (Q1, Q2, Q3) for data.'''
    return percentile(data, 25), percentile(data, 50), percentile(data, 75)


def five_number_summary(data):
    '''Return (min, Q1, median, Q3, max).'''
    q1, q2, q3 = quartiles(data)
    return min(data), q1, q2, q3, max(data)


salaries = [28000, 32000, 35000, 38000, 40000, 42000, 45000,
            48000, 50000, 55000, 60000, 65000, 72000, 250000]  # last one: CEO

lo, q1, med, q3, hi = five_number_summary(salaries)
iqr = q3 - q1

print(f"Salaries: {salaries}")
print()
print(f"  Minimum : Rs.{lo:,.0f}")
print(f"  Q1 (25%): Rs.{q1:,.0f}")
print(f"  Median  : Rs.{med:,.0f}")
print(f"  Q3 (75%): Rs.{q3:,.0f}")
print(f"  Maximum : Rs.{hi:,.0f}")
print(f"  IQR     : Rs.{iqr:,.0f}  (spread of the middle 50%)")
print()
print("Notice the mean would be pulled way up by the Rs.2,50,000 CEO salary.")
print("The median and IQR are far more representative of a 'typical' employee.")

---
## 4. IQR — Outlier Detection (Tukey's Method)

Lesson 02 detected outliers with the z-score method (|z| > 3). The **IQR method** (also called Tukey's fences) is an alternative that doesn't assume the data is normally distributed:

$$\text{Lower fence} = Q1 - 1.5 \times IQR$$
$$\text{Upper fence} = Q3 + 1.5 \times IQR$$

Any value outside these fences is flagged as an outlier. The multiplier 1.5 is the standard choice; 3.0 is used for "extreme" outliers.

In [ ]:
def find_outliers_iqr(data, k=1.5):
    '''
    Detect outliers using Tukey's IQR method.
    Returns list of (index, value) for values outside [Q1 - k*IQR, Q3 + k*IQR].
    '''
    q1, _, q3 = quartiles(data)
    iqr = q3 - q1
    lower_fence = q1 - k * iqr
    upper_fence = q3 + k * iqr
    return [(i, x) for i, x in enumerate(data) if x < lower_fence or x > upper_fence]


salaries = [28000, 32000, 35000, 38000, 40000, 42000, 45000,
            48000, 50000, 55000, 60000, 65000, 72000, 250000]

q1, _, q3 = quartiles(salaries)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = find_outliers_iqr(salaries)

print(f"Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={iqr:,.0f}")
print(f"Fences: [{lower_fence:,.0f}, {upper_fence:,.0f}]")
print()
print("Outliers found:")
for idx, val in outliers:
    print(f"  Index {idx}: Rs.{val:,.0f}")
print()
print("Why use IQR instead of z-score here? Z-score uses the MEAN and STD DEV —")
print("both of which are themselves distorted by the outlier. IQR is based on")
print("quartiles, which are far more resistant (robust) to extreme values.")

---
## 5. Visualising Spread — The ASCII Box Plot

In [ ]:
def box_plot(data, label="", width=60):
    '''Draw a simple ASCII box plot for data.'''
    lo, q1, med, q3, hi = five_number_summary(data)

    def pos(v):
        p = int((v - lo) / (hi - lo) * (width - 1)) if hi > lo else 0
        return max(0, min(width - 1, p))

    line = list("-" * width)
    p_lo, p_q1, p_med, p_q3, p_hi = pos(lo), pos(q1), pos(med), pos(q3), pos(hi)

    for i in range(p_q1, p_q3 + 1):
        line[i] = "="            # box (IQR)
    line[p_med] = "|"            # median
    line[p_lo] = "<"             # min whisker
    line[p_hi] = ">"             # max whisker

    print(f"  {label}")
    print(f"  |{''.join(line)}|")
    print(f"   min={lo:.0f}   Q1={q1:.0f}   median={med:.0f}   Q3={q3:.0f}   max={hi:.0f}")
    print()


exam_a = [60, 65, 68, 70, 72, 74, 75, 78, 80, 85, 90]
exam_b = [40, 55, 60, 65, 68, 70, 72, 75, 90, 95, 99]

box_plot(exam_a, "Exam A")
box_plot(exam_b, "Exam B")
print("< / > = min/max    = = the IQR box (middle 50%)    | = median")

---
## 6. Correlation — Do Two Variables Move Together?

**Correlation** measures whether two variables tend to increase or decrease together, and how strongly.

- Study hours ↑, exam score ↑ → **positive correlation**
- Price ↑, quantity sold ↓ → **negative correlation**
- Shoe size and IQ → **no correlation** (unrelated)

This is one of the most important tools in data science: before you build any predictive model, you check which features actually correlate with the target you're trying to predict.

In [ ]:
# Study hours vs exam scores — do they move together?
study_hours = [1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 8, 9]
exam_scores = [42, 50, 48, 55, 60, 63, 68, 72, 75, 80, 85, 92]

def ascii_scatter(x, y, width=50, height=15):
    '''Rough ASCII scatter plot.'''
    x_lo, x_hi = min(x), max(x)
    y_lo, y_hi = min(y), max(y)
    grid = [[" "] * width for _ in range(height)]

    for xi, yi in zip(x, y):
        col = int((xi - x_lo) / (x_hi - x_lo) * (width - 1)) if x_hi > x_lo else 0
        row = int((yi - y_lo) / (y_hi - y_lo) * (height - 1)) if y_hi > y_lo else 0
        grid[height - 1 - row][col] = "*"

    for row in grid:
        print("  |" + "".join(row) + "|")
    print(f"   low={x_lo} ... study hours ... high={x_hi}")

print("Study Hours vs Exam Score")
ascii_scatter(study_hours, exam_scores)
print()
print("The dots trend upward from bottom-left to top-right — a clear positive relationship.")

---
## 7. Computing Pearson Correlation Coefficient From Scratch

The **Pearson correlation coefficient (r)** quantifies the linear relationship between two variables:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2} \sqrt{\sum (y_i - \bar{y})^2}}$$

r always falls between **-1 and +1**:
- **r = +1** → perfect positive linear relationship
- **r = 0** → no linear relationship
- **r = -1** → perfect negative linear relationship

In [ ]:
import math

def mean(data):
    if not data:
        raise ValueError("Cannot compute mean of an empty list")
    return sum(data) / len(data)


def correlation(x, y):
    '''Compute the Pearson correlation coefficient between x and y.'''
    if len(x) != len(y):
        raise ValueError("x and y must be the same length")
    if len(x) < 2:
        raise ValueError("Need at least 2 data points")

    mx, my = mean(x), mean(y)
    numerator = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    denom_x = math.sqrt(sum((xi - mx) ** 2 for xi in x))
    denom_y = math.sqrt(sum((yi - my) ** 2 for yi in y))

    if denom_x == 0 or denom_y == 0:
        raise ValueError("Correlation is undefined when one variable has zero variance")

    return numerator / (denom_x * denom_y)


study_hours = [1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 8, 9]
exam_scores = [42, 50, 48, 55, 60, 63, 68, 72, 75, 80, 85, 92]

price       = [10, 12, 14, 16, 18, 20, 22, 24, 26, 28]
qty_sold    = [500, 470, 430, 410, 380, 340, 300, 270, 230, 200]

r_study = correlation(study_hours, exam_scores)
r_price = correlation(price, qty_sold)

print(f"Study hours vs exam score : r = {r_study:+.3f}  (strong positive)")
print(f"Price vs quantity sold    : r = {r_price:+.3f}  (strong negative)")

---
## 8. Interpreting Correlation — Strength, Direction, and the Causation Trap

| \|r\| range | Strength |
|---|---|
| 0.8 – 1.0 | Strong |
| 0.5 – 0.8 | Moderate |
| 0.2 – 0.5 | Weak |
| 0.0 – 0.2 | Negligible / none |

**Correlation is not causation.** Two variables can be strongly correlated without one causing the other — often because a hidden third factor (a *confounding variable*) drives both.

Classic example: ice cream sales and drowning deaths are strongly positively correlated. Does ice cream cause drowning? No — **summer heat** drives both: more people swim, and more people buy ice cream.

In [ ]:
def interpret_correlation(r):
    '''Return a plain-English interpretation of a correlation coefficient.'''
    if r == 0:
        return "no correlation"
    strength = "strong" if abs(r) >= 0.8 else "moderate" if abs(r) >= 0.5 else "weak" if abs(r) >= 0.2 else "negligible"
    direction = "positive" if r > 0 else "negative"
    return f"{strength} {direction} correlation"


# The classic spurious correlation: ice cream sales vs drowning incidents
# Both driven by a hidden factor: temperature (summer)
month_temp     = [10, 12, 18, 24, 30, 34, 35, 33, 28, 20, 14, 11]   # deg C, Jan-Dec
ice_cream_sold = [200, 220, 350, 500, 800, 1000, 1050, 980, 700, 400, 250, 210]
drownings      = [1, 1, 2, 4, 7, 10, 11, 9, 6, 3, 1, 1]

r_ice_drown  = correlation(ice_cream_sold, drownings)
r_temp_ice   = correlation(month_temp, ice_cream_sold)
r_temp_drown = correlation(month_temp, drownings)

print(f"Ice cream sales vs drownings : r = {r_ice_drown:+.3f}  ({interpret_correlation(r_ice_drown)})")
print(f"Temperature vs ice cream     : r = {r_temp_ice:+.3f}  ({interpret_correlation(r_temp_ice)})")
print(f"Temperature vs drownings     : r = {r_temp_drown:+.3f}  ({interpret_correlation(r_temp_drown)})")
print()
print("Ice cream doesn't cause drowning. Temperature drives BOTH.")
print("Whenever you find a strong correlation, ask: 'could a third factor explain this?'")

---
## 9. Correlation Doesn't Capture Everything — Nonlinear Relationships

In [ ]:
# A perfect relationship — but NOT linear
x = [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]
y = [v ** 2 for v in x]   # y = x^2, a perfect parabola

r = correlation(x, y)

print(f"x : {x}")
print(f"y : {y}  (y = x^2)")
print()
print(f"Pearson r = {r:+.3f}")
print()
print("Every point lies EXACTLY on a curve — a perfect relationship. But Pearson's r")
print("is close to 0, because r only detects LINEAR relationships. As x increases,")
print("y goes down then up — no consistent straight-line trend.")
print()
print("Lesson: always look at a scatter plot, not just the correlation number.")
print("A low r does not always mean 'no relationship' — it may mean 'not a LINEAR one'.")

---
## 10. Built-in Verification

In [ ]:
import statistics

data = [55, 60, 62, 65, 68, 70, 72, 75, 78, 80, 82, 85, 88, 90, 95]
x = [1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 8, 9]
y = [42, 50, 48, 55, 60, 63, 68, 72, 75, 80, 85, 92]

# statistics.quantiles(n=4) returns [Q1, Q2, Q3] using a slightly different method
# than our linear-interpolation percentile() — small differences are normal
stdlib_quartiles = statistics.quantiles(data, n=4)
our_quartiles    = quartiles(data)

print("Quartiles comparison:")
print(f"  Ours   : Q1={our_quartiles[0]:.2f}  Q2={our_quartiles[1]:.2f}  Q3={our_quartiles[2]:.2f}")
print(f"  stdlib : Q1={stdlib_quartiles[0]:.2f}  Q2={stdlib_quartiles[1]:.2f}  Q3={stdlib_quartiles[2]:.2f}")
print("  (Different libraries use different interpolation methods — small gaps are expected.)")
print()

stdlib_r = statistics.correlation(x, y)
our_r    = correlation(x, y)
print("Correlation comparison:")
print(f"  Ours   : r = {our_r:.6f}")
print(f"  stdlib : r = {stdlib_r:.6f}")
print(f"  Match  : {'match' if abs(our_r - stdlib_r) < 1e-9 else 'MISMATCH'}")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Confusing percentile with percentage --------------------------

class_scores = [40, 45, 50, 55, 60, 62, 65, 68, 70, 72, 75, 78, 80, 85, 90]
my_score = 65

my_percentile = sum(1 for x in class_scores if x < my_score) / len(class_scores) * 100

print("Mistake 1 — Percentile vs Percentage:")
print(f"  Raw score       : {my_score} (a PERCENTAGE — out of 100 marks)")
print(f"  Percentile rank : ~{my_percentile:.0f}th (a RANK — % of people you scored above)")
print()
print("  'I got a 65 percentile' is WRONG if you mean you scored 65%.")
print("  '65th percentile' means you outperformed 65% of test-takers —")
print("  your raw score could be anything depending on how hard the exam was.")

In [ ]:
# -- Mistake 2: Assuming all tools compute percentiles the same way -----------

data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# There are several documented percentile interpolation methods.
# Our method matches NumPy's default ('linear') and pandas' default.
p50_ours = percentile(data, 50)

print("Mistake 2 — Percentile methods differ across tools:")
print(f"  Our 50th percentile (linear interpolation): {p50_ours}")
print()
print("  NumPy default ('linear'), pandas .quantile() default, and Excel's")
print("  PERCENTILE.INC all use the SAME method as ours — good, they'll match.")
print("  But R's default quantile() method, and some other tools, use a")
print("  different interpolation rule and can give slightly different answers.")
print("  Always check which method a tool uses before comparing numbers across tools.")

In [ ]:
# -- Mistake 3: Reporting correlation as if it proves causation ---------------

# Number of firefighters sent to a fire vs damage caused ($)
firefighters = [5, 10, 15, 20, 25, 30, 35, 40]
damage       = [10000, 25000, 45000, 60000, 80000, 95000, 115000, 130000]

r = correlation(firefighters, damage)
print("Mistake 3 — Correlation implies causation:")
print(f"  Firefighters sent vs fire damage: r = {r:+.3f}  (strong positive)")
print()
print("  WRONG conclusion: 'Sending more firefighters causes more damage — send fewer!'")
print("  RIGHT conclusion: bigger fires need MORE firefighters AND cause MORE damage.")
print("  Fire SIZE is the confounding variable driving both. Never skip this check.")

In [ ]:
# -- Mistake 4: A single outlier can flip or inflate correlation --------------

x_clean = [1, 2, 3, 4, 5, 6, 7, 8]
y_clean = [2, 3, 5, 4, 6, 5, 7, 8]   # mild positive relationship

x_outlier = x_clean + [50]
y_outlier = y_clean + [1]            # one extreme point pulling the other way

r_clean   = correlation(x_clean, y_clean)
r_outlier = correlation(x_outlier, y_outlier)

print("Mistake 4 — Outliers distort Pearson correlation:")
print(f"  Without outlier : r = {r_clean:+.3f}")
print(f"  With ONE outlier: r = {r_outlier:+.3f}")
print()
print("  Just like mean and std dev, Pearson correlation is NOT robust to outliers.")
print("  Always plot your data before trusting a correlation number.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Percentile Rank Calculator

Write `percentile_rank(data, value)` that returns what percentile a given `value` falls at within `data` (percentage of values strictly below it). Test it on the dataset below for a student who scored 78.

```python
class_scores = [45, 50, 55, 58, 60, 62, 65, 68, 70, 72, 75, 78, 80, 85, 90, 95]
```

In [ ]:
class_scores = [45, 50, 55, 58, 60, 62, 65, 68, 70, 72, 75, 78, 80, 85, 90, 95]
# Your code here

### Exercise 2 — Delivery Time Outlier Detector

A food delivery app logs delivery times (minutes). Use the IQR method to flag outliers, and print the five-number summary.

```python
delivery_times = [18, 22, 20, 25, 19, 21, 23, 90, 24, 20, 22, 19, 18, 26, 5]
```

In [ ]:
delivery_times = [18, 22, 20, 25, 19, 21, 23, 90, 24, 20, 22, 19, 18, 26, 5]
# Your code here

### Exercise 3 — Marketing Correlation Report

You're a marketing analyst. Compute the correlation between social media ad spend and website signups. Write a function `correlation_report(x, y, x_label, y_label)` that prints r and its plain-English interpretation using `interpret_correlation()`.

```python
ad_spend = [200, 350, 400, 500, 600, 700, 800, 950, 1000, 1200]
signups  = [12, 22, 25, 30, 35, 38, 45, 52, 55, 65]
```

In [ ]:
ad_spend = [200, 350, 400, 500, 600, 700, 800, 950, 1000, 1200]
signups  = [12, 22, 25, 30, 35, 38, 45, 52, 55, 65]
# Your code here

### Exercise 4 — Box Plot Comparison Report

Write `compare_distributions(datasets)` where `datasets` is a dict of `{label: data}`. For each dataset, print the box plot (reuse `box_plot()`), and identify which dataset has the largest IQR.

```python
section_a = [55, 60, 62, 65, 68, 70, 72, 75, 78, 80]
section_b = [30, 45, 50, 65, 70, 75, 80, 85, 95, 99]
```

In [ ]:
section_a = [55, 60, 62, 65, 68, 70, 72, 75, 78, 80]
section_b = [30, 45, 50, 65, 70, 75, 80, 85, 95, 99]
datasets = {"Section A": section_a, "Section B": section_b}

def compare_distributions(datasets):
    # Your code here
    pass

compare_distributions(datasets)

### Exercise 5 — (Challenge) Correlation Matrix

You have a dataset of student records with multiple numeric features. Write `correlation_matrix(records, fields)` that:
1. Computes the pairwise Pearson correlation between every pair of fields
2. Prints the result as a formatted matrix (fields as rows and columns)
3. Identifies and prints the single strongest correlated pair (excluding a field with itself)

```python
students = [
    {"study_hours": 2, "attendance": 65, "score": 50},
    {"study_hours": 4, "attendance": 70, "score": 58},
    {"study_hours": 5, "attendance": 80, "score": 65},
    {"study_hours": 6, "attendance": 75, "score": 70},
    {"study_hours": 7, "attendance": 90, "score": 78},
    {"study_hours": 8, "attendance": 85, "score": 82},
    {"study_hours": 9, "attendance": 95, "score": 90},
    {"study_hours": 3, "attendance": 60, "score": 52},
]
```

In [ ]:
students = [
    {"study_hours": 2, "attendance": 65, "score": 50},
    {"study_hours": 4, "attendance": 70, "score": 58},
    {"study_hours": 5, "attendance": 80, "score": 65},
    {"study_hours": 6, "attendance": 75, "score": 70},
    {"study_hours": 7, "attendance": 90, "score": 78},
    {"study_hours": 8, "attendance": 85, "score": 82},
    {"study_hours": 9, "attendance": 95, "score": 90},
    {"study_hours": 3, "attendance": 60, "score": 52},
]

def correlation_matrix(records, fields):
    # Your code here
    pass

correlation_matrix(students, ["study_hours", "attendance", "score"])

---
## 📌 Key Takeaways

- **Percentiles tell you rank, not raw value.** A percentile answers "what fraction of the data falls below this point?" — it's why "92nd percentile" on a hard exam can represent a much lower raw score than "92nd percentile" on an easy one. Percentiles power everything from JEE ranks to "p95 latency" in engineering dashboards.

- **IQR is a robust alternative to z-scores for spread and outliers.** Because quartiles depend only on rank order (not on the mean or std dev), the IQR method for outlier detection (`Q1 - 1.5×IQR` to `Q3 + 1.5×IQR`) isn't distorted by the very outliers it's trying to detect — unlike the z-score method from Lesson 02.

- **Correlation measures linear association, not causation, and it's sensitive to outliers.** A high Pearson r tells you two variables move together in a straight-line pattern — nothing more. Always ask "could a third factor explain this?", check for nonlinear patterns with a scatter plot, and watch out for single extreme points inflating or flipping the result.

---
## What's Next?

**Lesson 04 — Basic Probability**
You've learned to describe data that already exists. Next, you'll flip to the forward-looking half of statistics: given what you know, how likely is a future outcome? Probability is the mathematical foundation behind every prediction a machine learning model makes.